# 06 — Optional external subject-indexed manifest gate

The internal experiment cannot establish unseen-person generalization because
GAVD provides no persistent subject identifier. This notebook only validates a
separately supplied, custodian-indexed external pose manifest. It does not
download media, infer identity, train a model, or manufacture an external
result. Subject identifiers must be provided under an approved data contract,
and subjects must be disjoint across train, validation, and test partitions.

Validation is fail-closed behind ethics, data-use, and derived-pose reviews
scoped specifically to the external dataset; GAVD reviews cannot authorize it.
If either required environment setting is absent, the validator is deliberately
not called and the status is **blocked / not run**. Setting an environment
variable cannot itself supply authorization. Passing this gate
validates structure and authorization prerequisites; it does not run an
external evaluation or constitute confirmation. Dataset labels, if
present outside this contract, remain annotations rather than diagnoses, and
no clinical claim follows from structural validation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
import os

from laterality.external import (
    ExternalEvaluationBlocked,
    ExternalManifestError,
    validate_external_manifest,
)
from laterality.visualization import external_gate_figure

manifest_setting = os.getenv("LATERALITY_EXTERNAL_MANIFEST")
external_governance_setting = os.getenv(
    "LATERALITY_EXTERNAL_GOVERNANCE"
)

if not manifest_setting or not external_governance_setting:
    external_status = {
        "status": "blocked / not run",
        "validator_called": False,
        "reason": (
            "Both LATERALITY_EXTERNAL_MANIFEST and an external-dataset-"
            "scoped LATERALITY_EXTERNAL_GOVERNANCE record are required."
        ),
        "evidence_created": False,
    }
else:
    pose_root_setting = os.getenv("LATERALITY_EXTERNAL_POSE_ROOT")
    try:
        external_cohort = validate_external_manifest(
            manifest_setting,
            external_governance_setting,
            pose_root=pose_root_setting,
        )
    except (ExternalEvaluationBlocked, ExternalManifestError) as error:
        external_status = {
            "status": "blocked / not run",
            "validator_called": True,
            "reason": str(error),
            "evidence_created": False,
        }
    else:
        external_status = {
            "status": "manifest contract validated; evaluation not run",
            "validator_called": True,
            "sequences": external_cohort.n_sequences,
            "subjects": external_cohort.n_subjects,
            "train_subjects": len(external_cohort.train_subject_ids),
            "validation_subjects": len(external_cohort.validation_subject_ids),
            "test_subjects": len(external_cohort.test_subject_ids),
            "evidence_created": False,
        }

show_inline(external_gate_figure(context, external_status))
external_status